# Batting Order Analysis (2026)
Identify where rostered batters hit in their real MLB lineups.

In [ ]:
import sys
import os
import time
import pandas as pd
import importlib
from datetime import datetime, timedelta

# Add parent dir so we can import mlb_processing
module_path = os.path.dirname(os.getcwd()) if 'fantasy_baseball' in os.getcwd() else os.getcwd()
if module_path not in sys.path:
    sys.path.insert(0, module_path)

from fantasy_baseball import mlb_processing as mp
importlib.reload(mp)  # Ensure we have the latest version of the module with global scraper support

## 1. Configuration
Set the season and date range for analysis.

In [ ]:
# ANALYSIS SETTINGS
SEASON_YEAR = 2026
NUM_DAYS_TO_LOOKBACK = 7  # Look at the past week for a good sample

end_dt = datetime.now()
start_dt = end_dt - timedelta(days=NUM_DAYS_TO_LOOKBACK)
num_days = (end_dt - start_dt).days + 1
dates = [(start_dt + timedelta(days=i)).strftime('%Y-%m-%d') for i in range(num_days)]

print(f"Analyzing Season: {SEASON_YEAR}")
print(f"Date Range: {dates[0]} to {dates[-1]} ({len(dates)} days)")

## 2. Load My Roster from ESPN

In [ ]:
# Setup league connection
config = mp.load_config()
league = mp.setup_league(config, year=SEASON_YEAR)
my_team_id = int(config['BASEBALL']['BB_MY_TEAM_ID'])

rosters = mp.get_league_rosters(league)
my_team = [p for p in rosters if p['team_id'] == my_team_id]

def is_pitcher(player):
    pitcher_keywords = {'SP', 'RP', 'P'}
    if player.get('player_position') in pitcher_keywords:
        return True
    slots = player.get('player_eligible_slots', [])
    hitter_slots = {'C', '1B', '2B', '3B', 'SS', 'OF', 'LF', 'CF', 'RF', 'DH'}
    if any(s in hitter_slots for s in slots):
        return False
    return any(s in pitcher_keywords for s in slots)

my_batters = [p for p in my_team if not is_pitcher(p)]
my_batter_names = [p['player_name'] for p in my_batters]

print(f"Team ID: {my_team_id}")
print(f"Batters found: {len(my_batters)}")
for p in my_batters[:5]:
    print(f"  {p['player_name']:25s} | {p['player_pro_team']}")
print("  ...")

## 3. Scrape Global Lineups from MLB.com
Scrape all games for each day.

In [ ]:
all_lineups = []
for dt in dates:
    print(f"Scraping all lineups for {dt}...", end=" ")
    try:
        lineup = mp.scrape_mlb_lineups(dt)
        if lineup:
            all_lineups.extend(lineup)
        print(f"{len(lineup)} entries found.")
    except Exception as e:
        print(f"FAILED: {e}")
    time.sleep(0.5)

if not all_lineups:
    print("No lineups were found. Please check your internet connection or MLB.com access.")
else:
    df_lineups = pd.DataFrame(all_lineups)
    print(f"\nTotal entries scraped: {len(df_lineups)}")

## 4. Match and Filter to My Roster

In [ ]:
if not all_lineups:
    df_my_lineups = pd.DataFrame()
else:
    df_lineups['roster_name'] = df_lineups['player_name'].apply(lambda x: mp.match_player_name(x, my_batter_names))
    df_my_lineups = df_lineups[df_lineups['roster_name'].notna()].copy()
    df_my_lineups = df_my_lineups.drop_duplicates(subset=['date', 'roster_name'], keep='first')
    
    print(f"Matched {df_my_lineups['roster_name'].nunique()} batters.")
    print(f"Total matched game-starts: {len(df_my_lineups)}")

## 5. Summary Statistics

In [27]:
if not df_my_lineups.empty:
    # 1. Calculate Averages
    avg_order = (
        df_my_lineups
        .groupby('roster_name')['batting_order']
        .agg(['mean', 'count'])
        .rename(columns={'mean': 'Avg_Ord', 'count': 'GS'})
    )
    
    # 2. Count at Each Batting Order Position
    order_counts = (
        df_my_lineups
        .groupby(['roster_name', 'batting_order'])
        .size()
        .unstack(fill_value=0)
    )
    
    # Ensure all 9 positions are columns for consistency
    for i in range(1, 10):
        if i not in order_counts.columns:
            order_counts[i] = 0
    order_counts = order_counts[sorted(order_counts.columns)]
    
    # 3. Create Summary Table
    res = avg_order.join(order_counts).round(2).sort_values('Avg_Ord')
    
    # 4. Map back Player Position and Team from Roster
    pos_map = {p['player_name']: p['player_position'] for p in my_batters}
    team_map = {p['player_name']: p['player_pro_team'] for p in my_batters}
    
    res['Pos'] = res.index.map(pos_map)
    res['Team'] = res.index.map(team_map)
    
    # Reorder columns
    cols = ['Team', 'Pos', 'Avg_Ord', 'GS'] + [i for i in range(1, 10)]
    res = res[cols]
    
    print("BATTING ORDER SUMMARY - MY ROSTER")
    display(res)
else:
    print("No matching data to display summary.")

BATTING ORDER SUMMARY - MY ROSTER


,Team,Pos,Avg_Ord,GS,1,2,3,4,5,6,7,8,9
roster_name,,,,,,,,,,,,,
George Springer,Tor,CF,1.00,6,6,0,0,0,0,0,0,0,0
Trent Grisham,NYY,1B/3B,1.00,5,5,0,0,0,0,0,0,0,0
Steven Kwan,Cle,2B/SS,1.00,7,7,0,0,0,0,0,0,0,0
Alex Bregman,ChC,SS,2.00,6,0,6,0,0,0,0,0,0,0
Shea Langeliers,Oak,1B,2.00,6,1,4,1,0,0,0,0,0,0
Hunter Goodman,Col,1B,2.00,6,0,6,0,0,0,0,0,0,0
Gleyber Torres,Det,3B,2.33,6,0,4,2,0,0,0,0,0,0
Bryce Harper,Phi,2B,3.00,6,0,0,6,0,0,0,0,0,0
Cody Bellinger,NYY,2B/SS,3.00,6,0,0,6,0,0,0,0,0,0
